In [ ]:
import json
import ast
from typing import List, Optional

# ======================== 1. Категории и средние высоты этажей ========================

CATEGORY_AVG_HEIGHTS = {
    "Жильё": 3.0,
    "Офисы и бизнес-центры": 3.5,
    "Торговля": 4.0,
    "Общественное питание": 3.5,
    "Сфера услуг": 3.5,
    "Промышленность и склады": 6.0,
    "Транспорт": 5.0,
    "Спорт и фитнес": 5.0,
    "Культура и отдых": 4.5,
    "Образование": 3.2,
    "Медицина": 3.2,
    "Религия": 8.0,
    "Государство и администрация": 3.5,
    "Инфраструктурные и малые архитектурные формы": 2.5,
    "Смешанные и многопрофильные комплексы": 4.0
}

# Словарь для маппинга тегов в категории (точные совпадения)
TAG_TO_CATEGORY = {
    # Жильё
    "жилое здание": "Жильё",
    "гостиница, хостел": "Жильё",
    "общежитие": "Жильё",
    # Офисы и бизнес-центры
    "офисы": "Офисы и бизнес-центры",
    "бизнес-центр": "Офисы и бизнес-центры",
    "коммерческая организация, фирма": "Офисы и бизнес-центры",
    "банк": "Офисы и бизнес-центры",
    # Торговля
    "торговля": "Торговля",
    "торговый центр, вещевой рынок": "Торговля",
    "продукты": "Торговля",
    "одежда, обувь, аксессуары": "Торговля",
    "техника, электроника": "Торговля",
    "мебель, интерьер": "Торговля",
    "строительный, хозяйственный магазин": "Торговля",
    "цветы, садоводство": "Торговля",
    "оптовая торговля": "Торговля",
    "книжный, канцтовары": "Торговля",
    "ювелирные изделия": "Торговля",
    "антиквариат, предметы искусства": "Торговля",
    "автозапчасти, автоаксессуары": "Торговля",
    "автосалон, авторынок": "Торговля",
    # Общественное питание
    "кафе, ресторан": "Общественное питание",
    "фастфуд, быстрое питание": "Общественное питание",
    "бар, пивная": "Общественное питание",
    "кофейня, чайная": "Общественное питание",
    "общественное питание, столовая": "Общественное питание",
    # Сфера услуг
    "бизнес и услуги": "Сфера услуг",
    "автосервис, автотехцентр": "Сфера услуг",
    "автомойка": "Сфера услуг",
    "шиномонтаж": "Сфера услуг",
    "салон красоты, парикмахерская": "Сфера услуг",
    "ателье, ремонт, бытовые услуги": "Сфера услуг",
    "химчистка, прачечная": "Сфера услуг",
    "ритуальные услуги": "Сфера услуг",
    "фото, полиграфические услуги": "Сфера услуг",
    "ветеринария": "Сфера услуг",
    "юридические услуги": "Сфера услуг",
    "почта, услуги связи": "Сфера услуг",
    "салон связи": "Сфера услуг",
    "пункт техосмотра": "Сфера услуг",
    "зоопарк, дельфинарий": "Сфера услуг",
    # Промышленность и склады
    "промышленность": "Промышленность и склады",
    "службы и объекты ЖКХ": "Промышленность и склады",
    "гаражный кооператив": "Промышленность и склады",
    "утилизация отходов": "Промышленность и склады",
    "штрафстоянка": "Промышленность и склады",
    # Транспорт
    "транспорт": "Транспорт",
    "вокзал": "Транспорт",
    "аэродром": "Транспорт",
    "морской порт, вокзал": "Транспорт",
    "заправочная станция": "Транспорт",
    "Парковка": "Транспорт",
    "велопарковка": "Транспорт",
    "кассы": "Транспорт",
    "выход станции скоростного транспорта": "Транспорт",
    "транспорт, перевозки": "Транспорт",
    # Спорт и фитнес
    "спорт": "Спорт и фитнес",
    "стадион, спорткомплекс": "Спорт и фитнес",
    "бассейн": "Спорт и фитнес",
    "фитнес, тренажёрный зал": "Спорт и фитнес",
    "спортплощадка": "Спорт и фитнес",
    "ипподром, конно-спортивные клубы": "Спорт и фитнес",
    "спортивный объект": "Спорт и фитнес",
    # Культура и отдых
    "культура и отдых": "Культура и отдых",
    "развлекательный центр": "Культура и отдых",
    "культурно-развлекательное учреждение": "Культура и отдых",
    "ночной клуб": "Культура и отдых",
    "парк развлечений, аттракционы": "Культура и отдых",
    "санаторий, дом отдыха": "Культура и отдых",
    # Образование
    "образование": "Образование",
    "школа, гимназия": "Образование",
    "детский сад, ясли": "Образование",
    "колледж, техникум": "Образование",
    "высшее образование": "Образование",
    "дополнительное образование, секция": "Образование",
    "научное учреждение, НИИ": "Образование",
    "автошкола": "Образование",
    # Медицина
    "медицина": "Медицина",
    "больница, поликлиника": "Медицина",
    "медицинское учреждение": "Медицина",
    "стоматология": "Медицина",
    "аптека": "Медицина",
    # Религия
    "религия": "Религия",
    "церковь": "Религия",
    "мечеть": "Религия",
    "синагога": "Религия",
    "пагода": "Религия",
    "религиозное учреждение": "Религия",
    # Государство и администрация
    "государство": "Государство и администрация",
    "орган власти, администрация": "Государство и администрация",
    "суд, прокуратура": "Государство и администрация",
    "правоохранительные органы, спецслужбы": "Государство и администрация",
    "пожарные, спасательные службы": "Государство и администрация",
    "военкомат, армия": "Государство и администрация",
    "исправительное учреждение": "Государство и администрация",
    "ЗАГС, дворец бракосочетания": "Государство и администрация",
    "налоговая инспекция": "Государство и администрация",
    "паспортная служба": "Государство и администрация",
    "госуслуги, служба одного окна": "Государство и администрация",
    "пенсионное и социальное обеспечение": "Государство и администрация",
    "посольство, консульство, визовый центр": "Государство и администрация",
    # Инфраструктурные и малые архитектурные формы
    "постройка, сооружение": "Инфраструктурные и малые архитектурные формы",
    "стилобат": "Инфраструктурные и малые архитектурные формы",
    "туалет": "Инфраструктурные и малые архитектурные формы",
    "беседка": "Инфраструктурные и малые архитектурные формы",
    "скамейка": "Инфраструктурные и малые архитектурные формы",
    "мусорная площадка": "Инфраструктурные и малые архитектурные формы",
    "вольер для птиц": "Инфраструктурные и малые архитектурные формы",
    # Смешанные и многопрофильные комплексы
    "Комплекс зданий": "Смешанные и многопрофильные комплексы",
}

def assign_category(tags: Optional[List[str]]) -> str:
    """Определяет категорию по списку тегов."""
    if not tags:
        return "Инфраструктурные и малые архитектурные формы"
    for tag in tags:
        if tag in TAG_TO_CATEGORY:
            return TAG_TO_CATEGORY[tag]
    # Эвристика: если есть слово "комплекс" – отнести к смешанным
    for tag in tags:
        if tag and "комплекс" in tag.lower():
            return "Смешанные и многопрофильные комплексы"
    return "Инфраструктурные и малые архитектурные формы"

def process_geojson(input_file: str, output_file: str):
    """Обрабатывает GeoJSON файл."""
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Проверяем структуру: должен быть FeatureCollection
    if data.get("type") != "FeatureCollection":
        raise ValueError("Файл не является GeoJSON FeatureCollection")

    new_features = []
    for feature in data["features"]:
        props = feature.get("properties", {})
        # Удаляем ненужные поля
        props.pop("Unnamed: 0", None)
        props.pop("gkh_address", None)
        # Удаляем дубликат площади, если есть
        props.pop("area_m2", None)

        # Обрабатываем теги
        raw_tags = props.get("tags", [])
        # Если теги пришли строкой (например, "['tag1', 'tag2']"), парсим
        if isinstance(raw_tags, str):
            try:
                raw_tags = ast.literal_eval(raw_tags)
            except:
                raw_tags = []
        if not isinstance(raw_tags, list):
            raw_tags = []
        # Очищаем от None и пустых строк
        raw_tags = [t for t in raw_tags if t is not None and t != ""]
        category = assign_category(raw_tags)
        props["category"] = category
        props.pop("tags", None)  # исходные теги больше не нужны

        # Этажность
        min_floor = props.get("gkh_floor_count_min")
        max_floor = props.get("gkh_floor_count_max")
        if max_floor is not None:
            floor_count = max_floor
        elif min_floor is not None:
            floor_count = min_floor
        else:
            floor_count = 1
        props.pop("gkh_floor_count_min", None)
        props.pop("gkh_floor_count_max", None)
        props["floor_count"] = floor_count

        # Средняя высота этажа и высота здания
        avg_height = CATEGORY_AVG_HEIGHTS.get(category, 3.0)
        props["avg_floor_height"] = avg_height
        props["building_height"] = floor_count * avg_height

        # Обновляем properties у feature
        feature["properties"] = props
        new_features.append(feature)

    # Сохраняем результат
    data["features"] = new_features
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Обработка завершена. Результат сохранён в {output_file}")

if __name__ == "__main__":
    input_file = "cleaned_buildings_A.geojson"
    output_file = "processed_buildings.geojson"
    process_geojson(input_file, output_file)